# 006_meeting_sender_unit_topic_group — 寄件人單位（PM / 業務）＋ 條目內容群（產品 / 客戶）

- 用途：讓會議紀錄能分「誰寄的」與「講什麼」兩個軸，不混在一起（2026-11-03，使用者決定）。
  1. **寄件人單位**：`mail_from` 第 4 段部門代碼的**前 4 碼** → `unit`（pm / sales / other）＋ `app`。種子表 `s_meeting_dept_unit`；mail 表補三欄 `forwarder_dept_prefix / forwarder_unit / forwarder_unit_basis`，本支順便把既有列全部回填（純規則、零 LLM）。
  2. **條目內容群**：task 3 的 14 個受控 `topic_tag` → `topic_group`（product / customer / other）。種子表 `s_meeting_topic_group`；002 的 `v_meeting_items_q` 多一欄 `topic_group`、`forwarder_unit`。條目表不加欄、不重萃。
- **正本是 [c05] `seed_data`**（隨本 notebook 進 git）。改對照 = 改 [c05] → 升 [c02] 版本 → 重跑本 notebook（先 `dry_run`）。拿掉的列標 `is_active = false`，不刪。
- **為什麼對「代碼前綴」不對「人」**（HANDOFF 2026-11-03 檢查項目）：人會換職務、同一人出現過兩種代碼；每封信都帶寄件當時的代碼，對代碼就自動跟著走。PM_Head `寄信檢討.md` 的名單只拿來**對答案**（tests / [c30] 的樣本），不寫進種子。
- 職務換了怎麼知道：換到種子已有的前綴 → 自動歸新單位；換到沒有的前綴 → `unknown`，task 1 [c12] 每週印 WARN、`meeting_extracted_pm_pack` C 節每月列「未登記前綴 / 同人代碼變了 / 新寄件人 / 停寄」；代碼不變職務變 → 資料看不出，靠每季對名單。
- 冪等：表 `IF NOT EXISTS`、`ALTER TABLE ADD COLUMNS` 只在缺欄時執行、種子 MERGE、回填是整表重算（代碼是寄信當時的值，重算 = 同一封信永遠得到「當時那個代碼」的單位）。每次正式執行寫一列 `b_meeting_migration_log`。
- 參數（widgets）：`catalog`、`schema`、`dry_run`（預設 `true`：只驗證、印 DDL 與種子差異）、`backfill`（預設 `true`：重算 mail 表三欄）
- 跑完要做：重跑 **002**（`dry_run = false`）讓 View 多出 `topic_group` / `forwarder_unit`；之後 task 1 每週自動填新信。
- 負責人 / 更新日期：（填）/ 2026-11-03

| 表 | 鍵 | 內容 |
|---|---|---|
| `s_meeting_dept_unit` | `dept_prefix` | 部門代碼前 4 碼 → `unit`（pm / sales / other）、`app`（TV / NB / MNT，可空）、`dept_name`（可空）、`note` |
| `s_meeting_topic_group` | `topic_tag` | 受控主題標籤 → `topic_group`（product / customer / other） |
| `s_meeting_record_mail_bu`（補欄） | — | `forwarder_dept_prefix`、`forwarder_unit`（pm / sales / other / unknown）、`forwarder_unit_basis`（dept_prefix / unknown / no_dept_code） |

In [ ]:
# [c01] params
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])
dbutils.widgets.dropdown("backfill", "true", ["true", "false"])  # 重算 mail 表既有列的三欄（純規則、零費用）

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
dry_run = dbutils.widgets.get("dry_run") == "true"
backfill = dbutils.widgets.get("backfill") == "true"
assert catalog and schema, "catalog / schema 不可為空"
print({"catalog": catalog, "schema": schema, "dry_run": dry_run, "backfill": backfill})

In [ ]:
# [c02] imports
# 本 cell 與 [c03]～[c05] 不碰 spark / dbutils，可被 tests/ 載入。
import csv
import io
import json
import re

SLUG = "006_meeting_sender_unit_topic_group"
UNIT = "bu"  # 單位代碼：表名後綴（規格 §3 命名規則）
COL_RE = r"^[a-z][a-z0-9_]*$"
TYPE_RE = r"^(STRING|BIGINT|INT|DOUBLE|BOOLEAN|DATE|TIMESTAMP|ARRAY<STRING>)$"

# 寄件人單位：pm（產品）/ sales（業務）/ other（IT、測試帳號、系統等既非 PM 也非業務）；mail 表另有 unknown = 前綴不在種子
SENDER_UNITS = ("pm", "sales", "other")
SENDER_UNIT_UNKNOWN = "unknown"
UNIT_BASIS = ("dept_prefix", "unknown", "no_dept_code")
APPS = ("TV", "NB", "MNT")
DEPT_PREFIX_LEN = 4           # 部門代碼前 4 碼（DTDB / DNBA …）；第 5–6 碼是課別，同單位內會變
DEPT_PREFIX_RE = r"^[A-Z0-9]{4}$"
# 條目內容群：product（產品：規劃、規格、時程、工程、品質、產能、成本、料況）/ customer（客戶：需求、allocation、價格、商務關係）/ other
TOPIC_GROUPS = ("product", "customer", "other")
# 同 task 3 [c02] TOPIC_TAGS（tests 會比：種子必須剛好涵蓋這 14 個）
TOPIC_TAGS = ("demand", "allocation", "price", "cost", "product_plan", "spec", "schedule", "supply_chain",
              "capacity", "quality", "engineering", "commercial_relation", "terminology", "admin")

DEPT_UNIT_VERSION = "0.1"     # 改 [c05] DEPT_UNIT_SEED 就升
TOPIC_GROUP_VERSION = "0.1"   # 改 [c05] TOPIC_GROUP_SEED 就升
DEPT_UNIT_COLS = ["dept_prefix", "unit", "app", "dept_name", "note"]
TOPIC_GROUP_COLS = ["topic_tag", "topic_group", "note"]

REQUIRED_COLS = {
    "s_meeting_dept_unit": ["dept_prefix", "unit", "is_active"],
    "s_meeting_topic_group": ["topic_tag", "topic_group", "is_active"],
}
# S3 欄位：只准落在 s_meeting_record_mail_bu / s_meeting_participants_bu；任何 View 不得暴露。
S3_COLS = {
    "display_name_raw", "name_zh", "name_en", "email", "emp_id", "account",
    "forwarder_name_zh", "forwarder_emp_id", "forwarder_account",
    "mail_to_raw", "mail_cc_raw", "mail_bcc_raw", "body_display",
}

In [ ]:
# [c03] pure_ddl
# spec → SQL 字串、種子驗證、前綴規則：純函式，不碰 spark。task 1 [c06] 與 tests 用同一條 dept_prefix 規則。
def q(s: str | None) -> str:
    """SQL 單引號字串常值；None → ''。"""
    return "'" + (s or "").replace("\\", "\\\\").replace("'", "\\'") + "'"


def build_create_table(fq: str, spec: dict) -> str:
    """CREATE TABLE IF NOT EXISTS；鍵欄 NOT NULL；表很小，不設 cluster。"""
    keys = set(spec["key"])
    cols = ",\n".join(
        f"    {name} {typ}{' NOT NULL' if name in keys else ''} COMMENT {q(comment)}"
        for name, typ, comment in spec["columns"]
    )
    return f"CREATE TABLE IF NOT EXISTS {fq} (\n{cols}\n) USING DELTA\nCOMMENT {q(spec['comment'])}"


def schema_ddl(spec: dict) -> str:
    """欄位 → 'a BIGINT, b STRING'（測試建空表用）。"""
    return ", ".join(f"{n} {t}" for n, t, _ in spec["columns"])


def add_columns_sql(fq: str, cols: list[tuple[str, str, str]]) -> str:
    """ALTER TABLE ADD COLUMNS（本支 [c21] 只對缺的欄執行）。"""
    parts = ", ".join(f"{n} {t} COMMENT {q(c)}" for n, t, c in cols)
    return f"ALTER TABLE {fq} ADD COLUMNS ({parts})"


def validate_specs(tables: dict) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。"""
    errs = []
    for t, spec in tables.items():
        names = [c[0] for c in spec["columns"]]
        dup = sorted({n for n in names if names.count(n) > 1})
        if dup:
            errs.append(f"{t} 欄位重複：{dup}")
        for n, typ, comment in spec["columns"]:
            if not re.match(COL_RE, n):
                errs.append(f"{t}.{n} 欄名格式不符（小寫英數底線）")
            if not re.match(TYPE_RE, typ):
                errs.append(f"{t}.{n} 型別不在允許清單：{typ}")
            if not comment:
                errs.append(f"{t}.{n} 沒有 COMMENT")
        for k in spec["key"]:
            if k not in names:
                errs.append(f"{t} key 欄 {k} 不存在")
        for r in REQUIRED_COLS.get(t, []):
            if r not in names:
                errs.append(f"{t} 缺必要欄 {r}")
        if not spec.get("comment"):
            errs.append(f"{t} 沒有表 COMMENT")
    return errs


# ---------- 種子 ----------
def parse_csv_text(text: str, expect_cols: list[str]) -> list[dict]:
    """CSV 格式文字 → list[dict]；欄位順序必須等於 expect_cols，值去頭尾空白，空字串 → None。（同 001 / 004）"""
    reader = csv.DictReader(io.StringIO(text.lstrip("﻿").strip()))
    if reader.fieldnames != expect_cols:
        raise ValueError(f"欄位應為 {expect_cols}，實際 {reader.fieldnames}")
    return [{k: (v.strip() or None) if isinstance(v, str) else v for k, v in r.items()} for r in reader]


def dept_prefix(dept_code: str | None) -> str | None:
    """部門代碼 → 比對鍵：去空白、大寫、取前 DEPT_PREFIX_LEN 碼；不足 4 碼或空 → None。task 1 [c06] 的 Spark 版本要同結果。"""
    s = (dept_code or "").strip().upper()
    return s[:DEPT_PREFIX_LEN] if len(s) >= DEPT_PREFIX_LEN else None


def validate_dept_seed(rows: list[dict]) -> list[str]:
    """回傳錯誤訊息；空 list = 通過。行號 = [c05] 該段第幾行（1 是欄名）。"""
    errs, seen = [], {}
    for i, r in enumerate(rows, start=2):
        p = r["dept_prefix"] or ""
        if not re.match(DEPT_PREFIX_RE, p):
            errs.append(f"DEPT_UNIT_SEED 第 {i} 行 dept_prefix 應為 4 碼大寫英數：{p!r}")
        if p in seen:
            errs.append(f"DEPT_UNIT_SEED 第 {i} 行 dept_prefix 重複（第 {seen[p]} 行）：{p}")
        seen[p] = i
        if r["unit"] not in SENDER_UNITS:
            errs.append(f"DEPT_UNIT_SEED 第 {i} 行 unit 不在 {list(SENDER_UNITS)}：{r['unit']!r}")
        if r["app"] is not None and r["app"] not in APPS:
            errs.append(f"DEPT_UNIT_SEED 第 {i} 行 app 不在 {list(APPS)}（或留空）：{r['app']!r}")
    return errs


def validate_topic_seed(rows: list[dict], expect_tags: tuple | set | None = None) -> list[str]:
    """種子必須剛好涵蓋 L1 詞表（task 3 [c02] TOPIC_TAGS）；group 只能三選一。"""
    errs, seen = [], {}
    for i, r in enumerate(rows, start=2):
        tag = r["topic_tag"] or ""
        if not re.match(COL_RE, tag):
            errs.append(f"TOPIC_GROUP_SEED 第 {i} 行 topic_tag 格式不符：{tag!r}")
        if tag in seen:
            errs.append(f"TOPIC_GROUP_SEED 第 {i} 行 topic_tag 重複：{tag}")
        seen[tag] = i
        if r["topic_group"] not in TOPIC_GROUPS:
            errs.append(f"TOPIC_GROUP_SEED 第 {i} 行 topic_group 不在 {list(TOPIC_GROUPS)}：{r['topic_group']!r}")
    if expect_tags is not None:
        missing = sorted(set(expect_tags) - set(seen))
        extra = sorted(set(seen) - set(expect_tags))
        if missing:
            errs.append(f"TOPIC_GROUP_SEED 少了 L1 詞表的 tag：{missing}")
        if extra:
            errs.append(f"TOPIC_GROUP_SEED 有 L1 詞表沒有的 tag：{extra}")
    return errs


def dept_unit_lookup(rows: list[dict]) -> dict[str, dict]:
    """dept_prefix → {unit, app, dept_name}。"""
    return {r["dept_prefix"]: {k: r[k] for k in ("unit", "app", "dept_name")} for r in rows}


def topic_group_lookup(rows: list[dict]) -> dict[str, str]:
    """topic_tag → topic_group。"""
    return {r["topic_tag"]: r["topic_group"] for r in rows}


def resolve_unit(dept_code: str | None, lookup: dict[str, dict]) -> tuple[str | None, str, str]:
    """(dept_prefix, unit, basis)：種子命中 → (prefix, unit, dept_prefix)；沒命中 → (prefix, unknown, unknown)；代碼空 → (None, unknown, no_dept_code)。"""
    p = dept_prefix(dept_code)
    if p is None:
        return None, SENDER_UNIT_UNKNOWN, "no_dept_code"
    hit = lookup.get(p)
    return (p, hit["unit"], "dept_prefix") if hit else (p, SENDER_UNIT_UNKNOWN, "unknown")

In [ ]:
# [c04] table_specs
# 正本。兩張種子表 ＋ mail 表要補的三欄（002 [c04] 的 mail spec 已同步列出這三欄；[c30] 比對用）。
TABLES = {
    "s_meeting_dept_unit": {
        "comment": "寄件人部門代碼前 4 碼 → 單位（pm / sales / other）與應用。task 1 每週查、006 回填既有信。正本：006 [c05] DEPT_UNIT_SEED；前綴不在表中的信 forwarder_unit = unknown，補一列再重跑 006 即回填。",
        "key": ["dept_prefix"],
        "columns": [
            ("dept_prefix", "STRING", "部門代碼前 4 碼（大寫），例 DTDB"),
            ("unit", "STRING", "pm（產品 PM）/ sales（業務）/ other（IT、測試帳號、系統等）"),
            ("app", "STRING", "這個前綴主要的應用：TV / NB / MNT；混用或不明留 NULL（只供報表分組，不用來推條目的 app）"),
            ("dept_name", "STRING", "部門名稱；待決 #13 沒有對照表前為 NULL"),
            ("note", "STRING", "依據（例：寄信檢討名單 2026-11-03）；標「待確認」的列是推測"),
            ("is_active", "BOOLEAN", "種子拿掉的列標 false，不刪"),
            ("updated_at", "TIMESTAMP", "最後異動時間"),
        ],
    },
    "s_meeting_topic_group": {
        "comment": "受控主題標籤 → 內容群（product 產品 / customer 客戶 / other）。v_meeting_items_q 的 topic_group 由此 join 主標籤；條目表不加欄。正本：006 [c05] TOPIC_GROUP_SEED。",
        "key": ["topic_tag"],
        "columns": [
            ("topic_tag", "STRING", "task 3 受控標籤（14 個）"),
            ("topic_group", "STRING", "product / customer / other"),
            ("note", "STRING", "為什麼歸這群"),
            ("is_active", "BOOLEAN", "種子拿掉的列標 false，不刪"),
            ("updated_at", "TIMESTAMP", "最後異動時間"),
        ],
    },
    "b_meeting_migration_log": {
        "comment": "會議萃取 migration 執行紀錄（只增不改）",
        "key": ["slug", "run_at"],
        "columns": [
            ("slug", "STRING", "migration 流水號＋名稱"),
            ("run_at", "TIMESTAMP", "執行時間"),
            ("run_by", "STRING", "執行者"),
            ("result", "STRING", "JSON：建表 / 種子 / 回填筆數"),
            ("note", "STRING", "備註"),
        ],
    },
}

# mail 表要補的欄（名稱, 型別, COMMENT）；順序 = 002 [c04] 追加的順序。
MAIL_ADD_COLS = [
    ("forwarder_dept_prefix", "STRING", "轉寄人部門代碼前 4 碼（大寫）；s_meeting_dept_unit 的比對鍵（006）"),
    ("forwarder_unit", "STRING", "轉寄人單位：pm / sales / other / unknown（前綴不在種子表）；由寄信當時的部門代碼查 s_meeting_dept_unit（006）"),
    ("forwarder_unit_basis", "STRING", "dept_prefix（種子命中）/ unknown（前綴未登記）/ no_dept_code（mail_from 無代碼）"),
]

In [ ]:
# [c05] seed_data
# 正本：改這裡 → 升 [c02] 版本 → 重跑本 notebook（先 dry_run；backfill = true 會把 mail 表既有列全部重算）。
# 格式同 CSV，第一行是欄名、逗號分隔、值不加引號。
#
# DEPT_UNIT_SEED：依 PM_Head 寄信檢討.md（2026-11-03）「預計需要寄信主管」名單 19 位寄件人的代碼對出來；名單內無例外。
#   第 4 碼在同一事業處內分 PM / 業務（DTD：B 業務、C / D PM；DNB：A 業務、B PM；DTV：A PM、B 業務），不是全公司一條規則，所以一前綴一列。
#   名單只拿來對答案，不寫人名；人員異動時這裡不用改，除非出現新前綴（task 1 [c12] / pm_pack C 節會列出來）。
#   沒放進來的前綴（Phase 0 曾出現：DTDA 111 封、DNB0 11、DITA 4、DGMA 1）→ unknown，等使用者確認再補列。
DEPT_UNIT_SEED = """
dept_prefix,unit,app,dept_name,note
DTDB,sales,MNT,,寄信檢討名單：MNT 業務 3 位（DTDBA0 / DTDBB0 / DTDBC0）
DTDC,pm,TV,,寄信檢討名單：TV PM 3 位（DTDCA0 / DTDCC0 / DTDCD0）
DTDD,pm,MNT,,寄信檢討名單：MNT PM 3 位（DTDDA0 / DTDDB0 / DTDDC0）
DNBA,sales,NB,,寄信檢討名單：NB 業務 2 位（DNBAA0 / DNBAB0）
DNBB,pm,NB,,寄信檢討名單：NB PM 6 位（DNBBA0～DNBBF0）
DTVA,pm,TV,,寄信檢討名單：TV PM 1 位（DTVAB0）；待確認（同前綴另有名單外 2 位）
DTVB,sales,TV,,寄信檢討名單：業務 1 位（DTVBA0），名單上應用空白、排在 TV 區塊；待確認
ABSB,other,,,內部測試帳號 / Drupal 轉寄測試（ABSBA0、ABSBB2）
"""

# TOPIC_GROUP_SEED：product = 產品面（PM 關心：規劃、規格、時程、工程、品質、產能、成本、料況）；customer = 客戶面（業務關心：需求、allocation、價格、商務關係）。
#   推測值（2026-11-03），使用者可改；cost 歸 product 是因為標籤定義是「材料或製造成本增減、cost adder」（Sales Brief 對照另有 §7.1 映射，不受影響）。
TOPIC_GROUP_SEED = """
topic_tag,topic_group,note
demand,customer,BP / FCST / 需求上下修 / 拉貨 / 市佔
allocation,customer,Allocation / 限簽 / 出貨餘量 / 備貨量
price,customer,報價 / 結價 / Deal / rebate / MDF
commercial_relation,customer,合約 / 合資 / 份額 / 對手動態 / 組織變動
product_plan,product,新機種 / 開案 / 停案 / EOL / line-up
spec,product,解析度 / 刷新率 / 介面 / POL / IC 選用
schedule,product,樣品 / MP / 驗證時程
supply_chain,product,缺料 / 交期 / 產地 / 供應商限制
capacity,product,廠別 / 轉廠 / 線體 / 產能
quality,product,不良率 / ppm / RA / 品質事件
engineering,product,實驗條件 / 製程參數 / 改機
cost,product,材料或製造成本增減 / cost adder（推測；要算客戶面就改 customer）
terminology,other,術語 / 口徑 / 換算規則
admin,other,會議行政
"""

In [ ]:
# [c10] build_ddl
# 驗 spec 與種子 → 組 DDL → 看前置表、缺欄、種子差異；dry_run 只印不建。
from pyspark.sql import functions as F

errors = validate_specs(TABLES)
dept_rows = parse_csv_text(DEPT_UNIT_SEED, DEPT_UNIT_COLS)
topic_rows = parse_csv_text(TOPIC_GROUP_SEED, TOPIC_GROUP_COLS)
errors += validate_dept_seed(dept_rows) + validate_topic_seed(topic_rows, TOPIC_TAGS)
for e in errors:
    print("ERROR", e)
assert not errors, f"spec / 種子驗證失敗 {len(errors)} 筆，修 [c04] / [c05] 後重跑"


def fq(name: str) -> str:
    return f"{catalog}.{schema}.{name}"


fq_mail = fq(f"s_meeting_record_mail_{UNIT}")
mail_exists = spark.catalog.tableExists(fq_mail)
mail_missing = [c for c in MAIL_ADD_COLS if mail_exists and c[0] not in spark.table(fq_mail).columns]
print("mail 表：", "存在" if mail_exists else "不存在（先跑 002）", "| 缺欄：", [c[0] for c in mail_missing] or "無")

table_ddl = {t: build_create_table(fq(t), s) for t, s in TABLES.items()}
existing = {t: spark.catalog.tableExists(fq(t)) for t in TABLES}
print("既有物件：", {k: v for k, v in existing.items() if v} or "無")
print("將建立：", [k for k, v in existing.items() if not v])


def _seed_df(rows: list[dict], cols: list[str]):
    return spark.createDataFrame([tuple(r[c] for c in cols) for r in rows], ", ".join(f"{c} string" for c in cols))


def _seed_diff(df_new, table: str, key: str, cmp: list[str]) -> dict:
    if not existing[table]:
        return {"insert": df_new.count(), "update": 0, "deactivate": 0}
    old = spark.table(fq(table))
    j = df_new.alias("n").join(old.alias("o"), key, "full_outer")
    same = F.lit(True)
    for c in cmp:
        same = same & F.col(f"n.{c}").eqNullSafe(F.col(f"o.{c}"))
    flags = {
        "insert": F.col("o.is_active").isNull() & F.col(f"n.{cmp[0]}").isNotNull(),
        "update": F.col(f"n.{cmp[0]}").isNotNull() & F.col("o.is_active").isNotNull() & (~same | ~F.col("o.is_active")),
        "deactivate": F.col(f"n.{cmp[0]}").isNull() & F.col("o.is_active"),
    }
    r = j.agg(*[F.sum(x.cast("int")).alias(k) for k, x in flags.items()]).collect()[0]
    return {k: int(r[k] or 0) for k in flags}


df_dept = _seed_df(dept_rows, DEPT_UNIT_COLS)
df_topic = _seed_df(topic_rows, TOPIC_GROUP_COLS)
seed_diff = {
    "dept_unit": _seed_diff(df_dept, "s_meeting_dept_unit", "dept_prefix", DEPT_UNIT_COLS[1:]),
    "topic_group": _seed_diff(df_topic, "s_meeting_topic_group", "topic_tag", TOPIC_GROUP_COLS[1:]),
}
print(f"種子 dept_unit {len(dept_rows)} 列（{DEPT_UNIT_VERSION}）、topic_group {len(topic_rows)} 列（{TOPIC_GROUP_VERSION}），預計變更：", seed_diff)

# 回填預覽：mail 表既有列依種子算出的單位分布 ＋ 未登記的前綴（只印代碼與封數，不印人名）
if mail_exists:
    lk = dept_unit_lookup(dept_rows)
    prefix_col = F.upper(F.substring(F.trim(F.col("forwarder_dept_code")), 1, DEPT_PREFIX_LEN))
    prefix_ok = F.length(F.trim(F.coalesce(F.col("forwarder_dept_code"), F.lit("")))) >= DEPT_PREFIX_LEN
    preview = (spark.table(fq_mail).select(F.when(prefix_ok, prefix_col).alias("dept_prefix"))
               .join(F.broadcast(df_dept.select("dept_prefix", "unit")), "dept_prefix", "left")
               .withColumn("unit", F.when(F.col("dept_prefix").isNull(), F.lit(SENDER_UNIT_UNKNOWN)).otherwise(F.coalesce("unit", F.lit(SENDER_UNIT_UNKNOWN)))))
    print("回填預覽 forwarder_unit 分布：", {r["unit"]: r["n"] for r in preview.groupBy("unit").agg(F.count("*").alias("n")).collect()})
    unknown_prefixes = [(r["dept_prefix"], r["n"]) for r in
                        preview.where(F.col("unit") == SENDER_UNIT_UNKNOWN).groupBy("dept_prefix").agg(F.count("*").alias("n"))
                        .orderBy(F.desc("n")).limit(20).collect()]
    print("未登記前綴（封數）top20：", unknown_prefixes or "無")

if dry_run:
    print("\ndry_run = true：只印 DDL 與種子，不建任何物件。確認後把 dry_run 改 false 重跑。\n")
    for ddl in table_ddl.values():
        print(ddl, end=";\n\n")
    if mail_missing:
        print(add_columns_sql(fq_mail, mail_missing), end=";\n\n")
    df_dept.show(len(dept_rows), truncate=False)
    df_topic.show(len(topic_rows), truncate=False)
else:
    assert mail_exists, f"{fq_mail} 不存在：先跑 002"

In [ ]:
# [c20] create_tables
# 一次性 DDL；IF NOT EXISTS 讓重跑無害。
if not dry_run:
    for t, ddl in table_ddl.items():
        spark.sql(ddl)
        print(("exists " if existing[t] else "created") + " " + fq(t))

In [ ]:
# [c21] alter_mail_table
# mail 表補三欄：只在缺欄時 ALTER（冪等）。002 [c04] 的 mail spec 已列這三欄，002 [c30] 之後比對不會再報缺欄。
# 用 ALTER 而不是 mergeSchema：表是 002 建的正式表，欄位變更要留在 migration 流水號裡可追。
if not dry_run and mail_missing:
    spark.sql(add_columns_sql(fq_mail, mail_missing))
    print("added columns", [c[0] for c in mail_missing], "→", fq_mail)
elif not dry_run:
    print("mail 表三欄已存在，略過 ALTER")

In [ ]:
# [c22] merge_seed
# 種子 MERGE：鍵相同且內容有變（或曾被停用）才更新 updated_at；種子拿掉的列標 is_active = false（不刪，保留回溯）。
def merge_seed(df_new, table: str, key: str, cols: list[str]) -> None:
    view = f"_seed_{table}"
    df_new.createOrReplaceTempView(view)
    cmp = [c for c in cols if c != key]
    differs = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in cmp) + " OR NOT t.is_active"
    spark.sql(f"""
        MERGE INTO {fq(table)} t USING {view} s ON t.{key} = s.{key}
        WHEN MATCHED AND ({differs}) THEN UPDATE SET
            {", ".join(f"t.{c} = s.{c}" for c in cmp)}, t.is_active = true, t.updated_at = current_timestamp()
        WHEN NOT MATCHED THEN INSERT ({", ".join(cols)}, is_active, updated_at)
            VALUES ({", ".join(f"s.{c}" for c in cols)}, true, current_timestamp())
        WHEN NOT MATCHED BY SOURCE AND t.is_active THEN UPDATE SET
            t.is_active = false, t.updated_at = current_timestamp()
    """)


if not dry_run:
    merge_seed(df_dept, "s_meeting_dept_unit", "dept_prefix", DEPT_UNIT_COLS)
    merge_seed(df_topic, "s_meeting_topic_group", "topic_tag", TOPIC_GROUP_COLS)
    print("seeds merged", seed_diff)

In [ ]:
# [c23] backfill_mail_unit
# 重算 mail 表每一列的三欄（整表 UPDATE，幾百到幾千封、零 LLM）。規則同 task 1 [c06] resolve_forwarder_unit：
# 前綴 = 寄信當時代碼的前 4 碼 → 查 active 種子 → 命中 unit / dept_prefix；沒命中 unknown / unknown；代碼不足 4 碼 unknown / no_dept_code。
# 整表重算而不是只補 NULL：種子改了（補前綴、改單位）重跑一次就全對；代碼本身是當時的值，重算不會改寫歷史。
backfill_result = {"skipped": True}
if not dry_run and backfill:
    spark.sql(f"""
        MERGE INTO {fq_mail} t
        USING (
            SELECT m.pk_seq,
                   CASE WHEN length(trim(coalesce(m.forwarder_dept_code, ''))) >= {DEPT_PREFIX_LEN}
                        THEN upper(substring(trim(m.forwarder_dept_code), 1, {DEPT_PREFIX_LEN})) END AS dept_prefix,
                   d.unit
            FROM {fq_mail} m
            LEFT JOIN (SELECT dept_prefix, unit FROM {fq('s_meeting_dept_unit')} WHERE is_active) d
              ON d.dept_prefix = CASE WHEN length(trim(coalesce(m.forwarder_dept_code, ''))) >= {DEPT_PREFIX_LEN}
                                      THEN upper(substring(trim(m.forwarder_dept_code), 1, {DEPT_PREFIX_LEN})) END
        ) s ON t.pk_seq = s.pk_seq
        WHEN MATCHED THEN UPDATE SET
            t.forwarder_dept_prefix = s.dept_prefix,
            t.forwarder_unit = coalesce(s.unit, '{SENDER_UNIT_UNKNOWN}'),
            t.forwarder_unit_basis = CASE WHEN s.dept_prefix IS NULL THEN 'no_dept_code'
                                          WHEN s.unit IS NULL THEN 'unknown' ELSE 'dept_prefix' END
    """)
    dist = spark.table(fq_mail).groupBy("forwarder_unit", "forwarder_unit_basis").agg(F.count("*").alias("n")).collect()
    backfill_result = {f"{r['forwarder_unit']}/{r['forwarder_unit_basis']}": r["n"] for r in dist}
    print("backfilled", backfill_result)
elif not dry_run:
    print("backfill = false：mail 表既有列不重算（新信由 task 1 填）")

In [ ]:
# [c24] write_log
# 每次正式執行（dry_run = false）寫一列 b_meeting_migration_log；只 INSERT，不改舊列。
if not dry_run:
    result = {
        "tables": {t: ("exists" if existing[t] else "created") for t in TABLES},
        "mail_columns_added": [c[0] for c in mail_missing],
        "seed": {"dept_unit_version": DEPT_UNIT_VERSION, "topic_group_version": TOPIC_GROUP_VERSION, **seed_diff},
        "backfill": backfill_result,
    }
    spark.sql(
        f"INSERT INTO {fq('b_meeting_migration_log')} "
        "SELECT :slug, current_timestamp(), current_user(), :result, :note",
        args={"slug": SLUG, "result": json.dumps(result, ensure_ascii=False), "note": None},
    )
    print("logged", result)

In [ ]:
# [c30] check_target
# 種子 active 列數 = [c05] 列數；mail 表三欄都在；unknown 的前綴列出來（只有代碼與封數，無人名）；印對答案用的樣本。
if not dry_run:
    problems = []
    for t, spec in TABLES.items():
        actual = {f.name for f in spark.table(fq(t)).schema.fields}
        missing = [n for n, _, _ in spec["columns"] if n not in actual]
        if missing:
            problems.append(f"{t} 缺欄 {missing}")
    n_dept = spark.table(fq("s_meeting_dept_unit")).where("is_active").count()
    n_topic = spark.table(fq("s_meeting_topic_group")).where("is_active").count()
    if n_dept != len(dept_rows):
        problems.append(f"s_meeting_dept_unit active {n_dept} ≠ 種子 {len(dept_rows)}")
    if n_topic != len(topic_rows):
        problems.append(f"s_meeting_topic_group active {n_topic} ≠ 種子 {len(topic_rows)}")
    mail_cols = spark.table(fq_mail).columns
    still_missing = [c[0] for c in MAIL_ADD_COLS if c[0] not in mail_cols]
    if still_missing:
        problems.append(f"{fq_mail} 仍缺欄 {still_missing}")
    for p in problems:
        print("PROBLEM", p)
    assert not problems, f"{len(problems)} 個問題"
    print({"dept_unit_active": n_dept, "topic_group_active": n_topic, "mail_cols": len(mail_cols)})
    mail = spark.table(fq_mail)
    print("forwarder_unit 分布（全表）：")
    mail.groupBy("forwarder_unit", "forwarder_unit_basis").agg(F.count("*").alias("mails"), F.countDistinct("forwarder_emp_id").alias("senders")
                                                             ).orderBy(F.desc("mails")).show(20, truncate=False)
    print("未登記前綴（補進 [c05] DEPT_UNIT_SEED 後重跑本支即回填）：")
    (mail.where(F.col("forwarder_unit") == SENDER_UNIT_UNKNOWN)
     .groupBy("forwarder_dept_prefix", "forwarder_dept_code").agg(F.count("*").alias("mails"), F.countDistinct("forwarder_emp_id").alias("senders"),
                                                                F.min("mail_date").alias("first_mail"), F.max("mail_date").alias("last_mail"))
     .orderBy(F.desc("mails")).show(30, truncate=False))
    print("對答案用：每個前綴最近一封的寄件人帳號（拿 PM_Head 寄信檢討.md 名單核對 unit / app 是否正確；帳號不是姓名、不入庫）：")
    from pyspark.sql.window import Window

    (mail.where(F.col("forwarder_dept_prefix").isNotNull())
     .withColumn("rn", F.row_number().over(Window.partitionBy("forwarder_dept_prefix").orderBy(F.desc("mail_date"), F.desc("pk_seq"))))
     .where("rn <= 2").select("forwarder_dept_prefix", "forwarder_unit", "forwarder_dept_code", "forwarder_account", "mail_date")
     .orderBy("forwarder_dept_prefix", F.desc("mail_date")).show(40, truncate=False))